# Câu 3c. Xây dựng vector đặc trưng văn bản: TF-IDF
**Input:** `../Data_Preprocessing` (kết quả Câu 1), mỗi file là 1 văn bản
**Output:** ma trận TF-IDF (số văn bản × số từ), lưu file `TFIDF.csv`
*(Tham khảo Ví dụ 4.4a, 4.4b – Chương 4)*

Công thức theo slide:
- TF(wk, di) = Tổng số lần xuất hiện của từ wk trong văn bản di / Tổng số từ trong văn bản di
- IDF(wk, D) = log10( Tổng số văn bản trong tập D / Số văn bản trong D có chứa từ wk )
- TF-IDF(wk, di, D) = TF(wk, di) × IDF(wk, D)

In [ ]:
# pip install scikit-learn pandas
import os
import math
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

## Đọc dữ liệu

In [ ]:
path_in = "../Data_Preprocessing"   # kết quả tiền xử lý của Câu 1

List_Files = sorted([f for f in os.listdir(path_in) if f.endswith(".txt")])
D = []          # tập văn bản (corpus), mỗi file là 1 văn bản
Doc_Names = []  # tên file tương ứng
for file_name in List_Files:
    f = open(os.path.join(path_in, file_name), "r", encoding="utf-8")
    text = " ".join(f.read().split())   # nối các câu (mỗi dòng 1 câu) thành 1 văn bản
    f.close()
    if text != "":                      # bỏ file rỗng (toàn bộ là stop words)
        D.append(text)
        Doc_Names.append(file_name)

print("Số văn bản:", len(D))
print("Văn bản đầu tiên (" + Doc_Names[0] + "):", D[0])

## Cách 1: Python thuần, đúng công thức slide (Ví dụ 4.4a)

In [ ]:
# Bước 1: Xây dựng tập từ vựng
def create_dict(doc_list):
    W = {}
    for d in doc_list:
        bow = d.split(" ")  # Tokenization
        W = set(W).union(set(bow))
    return W

# Bước 2.1: Đếm số lần xuất hiện của các từ trong văn bản
def WCount(d, W):
    wordDict = dict.fromkeys(W, 0)
    for word in d.split():
        wordDict[word] += 1
    return wordDict

# Bước 2.2 + 2.3: Tính TF
def compute_TF(W, d):
    tf_dict = {}
    bow_count = len(d.split())
    for word, count in W.items():
        tf_dict[word] = count / float(bow_count)
    return tf_dict

# Bước 3: Tính IDF
def compute_IDF(doc_list):
    idf_dict = {}
    N = len(doc_list)
    # Đếm số văn bản có chứa từ
    idf_dict = dict.fromkeys(doc_list[0].keys(), 0)
    for doc in doc_list:
        for word, count in doc.items():
            if count > 0:
                idf_dict[word] += 1
    for word, count in idf_dict.items():
        idf_dict[word] = math.log10(N / float(count))
    return idf_dict

# Bước 4: Tính TF-IDF
def compute_TFIDF(tf_bow, idfs):
    tfidf = {}
    for word, val in tf_bow.items():
        tfidf[word] = val * idfs[word]
    return tfidf

In [ ]:
# Bước 1
W = create_dict(D)
print("Số từ trong tập từ vựng:", len(W))

# Bước 2.1
List_WCount = [WCount(d, W) for d in D]

# Bước 2.2 + 2.3: TF
List_TF = [compute_TF(List_WCount[i], D[i]) for i in range(len(D))]

# Bước 3: IDF
idfs = compute_IDF(List_WCount)

# Bước 4: TF-IDF
List_TFIDF = [compute_TFIDF(tf, idfs) for tf in List_TF]

# Vector TF-IDF (cột sắp xếp theo alphabet)
df_tfidf_1 = pd.DataFrame(List_TFIDF, index=Doc_Names)
df_tfidf_1 = df_tfidf_1[sorted(df_tfidf_1.columns)]
print("Kích thước ma trận TF-IDF:", df_tfidf_1.shape)
df_tfidf_1.head().round(3)

In [ ]:
# Bảng IDF của các từ (từ hiếm -> IDF cao, từ phổ biến -> IDF thấp)
df_idf = pd.DataFrame(sorted(idfs.items(), key=lambda x: x[1]), columns=["Từ", "IDF"])
print("10 từ có IDF thấp nhất (xuất hiện ở nhiều văn bản):")
print(df_idf.head(10).round(3))

# 10 từ quan trọng nhất (TF-IDF cao nhất) của văn bản đầu tiên
print()
print("10 từ có TF-IDF cao nhất trong văn bản", Doc_Names[0])
print(df_tfidf_1.iloc[0].sort_values(ascending=False).head(10).round(3))

In [ ]:
df_tfidf_1.round(3).to_csv("TFIDF.csv", encoding="utf-8-sig")
print("Đã lưu TFIDF.csv")

## Cách 2: Sử dụng thư viện sklearn (Ví dụ 4.4b)
**Lưu ý:** `TfidfVectorizer` cho kết quả **khác** Cách 1 vì sklearn tính khác công thức slide:
- IDF = ln((1 + N) / (1 + df)) + 1 (dùng log tự nhiên, có làm trơn, cộng thêm 1)
- TF là số lần xuất hiện (không chia cho tổng số từ)
- Mỗi vector được chuẩn hóa L2 (độ dài = 1)

Vì vậy kết quả chính lấy theo Cách 1 (đúng công thức slide).

In [ ]:
tr_idf_model = TfidfVectorizer(token_pattern=r"(?u)\S+", lowercase=False)
tf_idf_vector = tr_idf_model.fit_transform(D)

tf_idf_array = tf_idf_vector.toarray()
W_2 = tr_idf_model.get_feature_names_out()

df_tfidf_2 = pd.DataFrame(tf_idf_array, columns=W_2, index=Doc_Names)
print("Kích thước ma trận TF-IDF:", df_tfidf_2.shape)
df_tfidf_2.head().round(3)

In [ ]:
# So sánh 10 từ quan trọng nhất của văn bản đầu tiên theo 2 cách
compare = pd.DataFrame({
    "Cách 1 (slide)": df_tfidf_1.iloc[0].sort_values(ascending=False).head(10).index,
    "Cách 2 (sklearn)": df_tfidf_2.iloc[0].sort_values(ascending=False).head(10).index})
compare